### Estimasi kebutuhan Ram untuk cache dataset di Ram saat training

In [ ]:
import os

def count_images(folder_path, extensions=(".jpg", ".jpeg", ".png")):
    count = 0
    for root, _, files in os.walk(folder_path):
        for file in files:
            if file.lower().endswith(extensions):
                count += 1
    return count

def estimate_yolo_ram_usage(num_images, image_size, dtype_bytes=4, channels=3, margin=0.5):
    bytes_per_image = image_size * image_size * channels * dtype_bytes
    total_bytes = num_images * bytes_per_image
    total_gb = total_bytes / (1024 ** 3)
    total_with_margin = total_gb * (1 + margin)
    return total_gb, total_with_margin

def recommend_cache_setting(required_gb, available_gb):
    if required_gb <= available_gb:
        return "✅ Aman menggunakan cache=True"
    else:
        return "⚠️ Gunakan cache='disk' atau kurangi resolusi / jumlah gambar"

if __name__ == "__main__":
    # ==== 🔧 GANTI SESUAI KEBUTUHAN ====
    dataset_path = r"F:\Kuliah\Skripsi\Code\river-trash-monitoring\dataset_75_15_10_augmented_3x\train\images"
    available_ram_gb = 10  # RAM kosong di sistem kamu
    dtype_bytes = 4        # float32
    margin = 0.5           # 50% safety margin
    channels = 3           # RGB
    resolutions = [416, 512, 640, 768]
    # ==================================

    num_images = count_images(dataset_path)
    print(f"\n📁 Jumlah gambar ditemukan: {num_images} di folder {dataset_path}\n")

    print("📊 Estimasi kebutuhan RAM (per resolusi):\n")
    for res in resolutions:
        base_gb, with_margin_gb = estimate_yolo_ram_usage(
            num_images, res, dtype_bytes, channels, margin
        )
        recommendation = recommend_cache_setting(with_margin_gb, available_ram_gb)
        print(f"- Resolusi {res}x{res}:")
        print(f"  ➤ Tanpa margin: {base_gb:.2f} GB")
        print(f"  ➤ Dengan margin: {with_margin_gb:.2f} GB")
        print(f"  ➤ Rekomendasi: {recommendation}\n")